In [2]:
import pandas as pd

In [3]:
# Load all datasets
customers = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\customers_raw.csv")
products = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\products_raw.csv")
orders = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\orders_raw.csv")
order_items = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\order_items_raw.csv")
delivery = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\delivery_raw.csv")
reviews = pd.read_csv(r"C:\Users\srava\Desktop\excel csv files\Revenue Leakage Datasets\reviews_raw.csv")

In [4]:
# fix data types
customers["signup_date"] = pd.to_datetime(customers["signup_date"],errors="coerce")
orders["order_date"] = pd.to_datetime(orders["order_date"],errors="coerce")
delivery["ship_date"] = pd.to_datetime(delivery["ship_date"],errors="coerce")
delivery["delivery_date"] = pd.to_datetime(delivery["delivery_date"],errors="coerce")
reviews["review_date"] = pd.to_datetime(reviews["review_date"],errors="coerce")

In [5]:
# duplicated values
customers.duplicated().sum()

np.int64(50)

In [6]:
orders.duplicated().sum()

np.int64(100)

In [7]:
# Remove duplicated records
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()

In [8]:
### Handle Missing Values
# customers table
customers["email"] = customers["email"].fillna("unknown@gmail.com")
customers["phone"] = customers["phone"].fillna("Not Provided")
customers["segment"] = customers["segment"].fillna("Unknown")
# Products table
products["sub_category"] = products["sub_category"].fillna("Unknown")
products["cost_price"] = products["cost_price"].fillna(products["cost_price"].median())
# orders table
orders["payment_type"] = orders["payment_type"].fillna("Not Specified")
# order items table
order_items["discount"] = order_items["discount"].fillna(0)
# delivery table
delivery["delivery_partner"] = delivery["delivery_partner"].fillna("Unknown")
# reviews table
reviews["review_text"] = reviews["review_text"].fillna("No Review")

In [9]:
# Standardize text columns
products["category"] = products["category"].replace({"Electrnics": "Electronics"})

In [10]:
# Normalize text case
orders["order_status"] = orders["order_status"].str.title()

In [11]:
# Fix Invalid Numeric Values
order_items["is_return"] = order_items["quantity"]<0
reviews.loc[reviews["rating"]>5,"rating"]=5

In [12]:
# where delivery date is before ship date
delivery_errors = delivery[delivery["delivery_date"] < delivery["ship_date"]]
print("Percentage of invalid delivery errors:",len(delivery_errors)/len(delivery)*100)

Percentage of invalid delivery errors: 13.112499999999999


In [13]:
# the error difference between those dates
delivery_errors["error_days"] = (delivery_errors["delivery_date"]-delivery_errors["ship_date"]).dt.days
delivery_errors["error_days"].describe()

C:\Users\srava\AppData\Local\Temp\ipykernel_9868\1618854364.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  delivery_errors["error_days"] = (delivery_errors["delivery_date"]-delivery_errors["ship_date"]).dt.days


count    1049.000000
mean       -1.527169
std         0.499499
min        -2.000000
25%        -2.000000
50%        -2.000000
75%        -1.000000
max        -1.000000
Name: error_days, dtype: float64

In [14]:
delivery_errors["error_days"].value_counts()

error_days
-2    553
-1    496
Name: count, dtype: int64

In [15]:
# Replace with missing values instead of removing 
delivery.loc[delivery["delivery_date"] < delivery["ship_date"],"delivery_date"] = pd.NaT

In [16]:
# Merge tables
data = order_items.merge(products,on="product_id",how="left")
data = data.merge(orders,on="order_id",how="left")
data = data.merge(customers,on="customer_id",how="left")
data = data.merge(reviews,on="order_id",how="left")
data = data.merge(delivery,on="order_id",how="left")

In [17]:
# revenue
data["revenue"] = data["selling_price"]*data["quantity"]

In [18]:
# cost
data["cost"] = data["cost_price"]*data["quantity"]

In [19]:
# profit
data["profit"] = data["revenue"]-data["cost"]-data["shipping_cost"]

In [20]:
# profit margin
data["profit_margin"] = data["profit"]/data["revenue"]

In [21]:
# delivery days
data["delivery_days"] = (data["delivery_date"]-data["ship_date"]).dt.days

In [22]:
# summary statistics
data.describe()

,quantity,selling_price,discount,shipping_cost,cost_price,order_date,order_value,signup_date,rating,review_date,ship_date,delivery_date,revenue,cost,profit,profit_margin,delivery_days
count,18155.000000,18155.000000,18155.000000,18155.000000,18155.000000,18155,18155.000000,18155,10014.000000,10014,17921,13937,18155.000000,18155.000000,18155.000000,18155.000000,13937.000000
mean,1.221316,1044.373396,0.119631,184.792068,542.061966,2025-09-04 21:55:37.857339648,2594.218783,2024-09-07 12:52:04.296337152,3.316856,2025-09-02 12:01:52.162971904,2025-09-07 23:58:09.113330944,2025-09-11 23:55:33.429001984,1286.050124,660.777472,440.480584,0.091369,5.001148
min,-1.000000,100.000000,0.000000,50.000000,51.000000,2025-03-08 00:00:00,200.000000,2023-03-10 00:00:00,1.000000,2025-03-08 00:00:00,2025-03-08 00:00:00,2025-03-08 00:00:00,-2000.000000,-998.000000,-2966.000000,-10.664062,0.000000
25%,-1.000000,559.000000,0.000000,50.000000,362.000000,2025-06-05 00:00:00,1369.000000,2023-12-09 00:00:00,2.000000,2025-06-05 00:00:00,2025-06-10 00:00:00,2025-06-14 00:00:00,-156.500000,-85.000000,-534.000000,-0.043218,2.000000
50%,1.000000,1044.000000,0.100000,100.000000,551.000000,2025-09-08 00:00:00,2611.000000,2024-09-02 00:00:00,3.000000,2025-09-01 00:00:00,2025-09-08 00:00:00,2025-09-12 00:00:00,1166.000000,620.000000,193.000000,0.445383,5.000000
75%,2.000000,1518.000000,0.200000,500.000000,741.000000,2025-12-01 00:00:00,3785.000000,2025-06-04 00:00:00,5.000000,2025-12-01 00:00:00,2025-12-08 00:00:00,2025-12-12 00:00:00,2476.000000,1300.000000,1208.000000,0.683210,8.000000
max,3.000000,2000.000000,0.300000,500.000000,998.000000,2026-03-08 00:00:00,4998.000000,2026-03-08 00:00:00,5.000000,2026-03-08 00:00:00,2026-03-08 00:00:00,2026-03-18 00:00:00,6000.000000,2994.000000,5662.000000,5.029126,10.000000
std,1.480823,548.112894,0.116791,185.556028,253.262905,NaN,1395.094231,NaN,1.513005,NaN,NaN,NaN,1867.617938,937.521349,1392.568712,1.142801,3.170187


In [98]:
data.to_csv("ecommerce_clean_dataset.csv",index=False)

In [27]:
data["discount"].value_counts()

discount
0.0    7332
0.2    3662
0.3    3617
0.1    3544
Name: count, dtype: int64